# NB01 - Utilisation des fonctionnalités de Delta Lake avec Databricks SQL

Obectifs : 
- Fonctionnlités : Time Travel, Historique de version et management de métadonnées
- SQL : `DESCRIBE EXTENDED`, `DESCRIBE HISTORY`, `VERSION AS OF` et `RESTORE TABLE`

## Mise en place
Le notebook dans `Resources/NB01/Setup` prend le fichier CSV **sales_data.csv** et le copie dans le volume `demo_<username>.demo_delta_lake.dld_demo`.


In [0]:
%run "./Resources/NB01/Setup"

On va maintenant créer la table `retail_sales` :

In [0]:
spark.sql("DROP TABLE IF EXISTS retail_sales")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS retail_sales
USING DELTA
AS 
SELECT *
FROM read_files(
  '{sales_path}',
  format => 'csv',
  header => 'true',
  inferSchema => 'true',
  delimiter => ';'
)
""")

display(spark.table("retail_sales"))

## Exploration

### Métadonnées

In [0]:
%sql
DESCRIBE EXTENDED retail_sales;

### Historique

In [0]:
%sql
DESCRIBE HISTORY retail_sales;

On voit que la version est **0** ce qui représente la création de la table, toutes prochaines modification de la table entrainera une incrémentation du numéro de version.

Avec la commande `DESCRIBE HISTORY` les métadonnées suivantes sont disponibles :
- **Version** : version de la table
- **Timestamp** : Timestamp de l'opération
- **Operation** : Type de modification (ex. `INSERT`, `UPDATE`, ...)
- **Operation Metrics** : nombre de lignes affectées, taille, ...
- **User Information** : Auteur de l'opération
- **Cluster Information** : ID du cluster qui à exécuté l'opération

## Simulation d'un changement et Time Travel

On va mettre à jour la table `retail_sales` pour ensuite visualiser les changements sur les métadonnées de la table et finalement requêter un ancienne version de la table en utilisant le **Time Travel**.

In [0]:
%sql
-- copier le timestamp dans le presse-papier pour pouvoir le coller dans le notebook python
select current_timestamp();

In [0]:
%sql
update retail_sales 
    set status = 'Delivered' 
    where status = 'Shipped';

In [0]:
%sql
insert into retail_sales 
    values ('ORD-031','2024-07-27','Mia Jensen','Denmark','Wireless Charger','Accessories',7,29.99,209.93,'Delivered', null);

In [0]:
%sql
DELETE FROM retail_sales where Order_ID = 'ORD-031';

Delta Lake supporte plusieurs opérations pour la gestion des données :

- **CREATE** : Création de tables Delta.
- **WRITE/INSERT** : Ajout de nouvelles données dans la table.
- **UPDATE** : Modification des données existantes selon des conditions.
- **DELETE** : Suppression de lignes spécifiques.
- **MERGE** : Fusion de données (upsert) entre deux tables.
- **OPTIMIZE** : Compactage des fichiers pour améliorer les performances.
- **RESTORE** : Restauration d'une version précédente de la table.
- **TIME TRAVEL** : Consultation de l’historique ou d’une version antérieure.
- **DESCRIBE HISTORY** : Visualisation des métadonnées et de l’historique des opérations.

Chaque opération est enregistrée dans l’historique Delta, permettant un suivi précis des modifications.

In [0]:
%sql
DESCRIBE HISTORY retail_sales;

On voit maintenant que la table à plusieurs versions, on va :
- Récupérer les données d'une version spécifique
- Récupérer les données à partir d'un **timestamp**
- Restaurer une table à partir d'une version antérieur

In [0]:
%sql
-- on récupére les lignes de la table avant la suppréssion de la dernière ligne inséré (Order_id = ORD-031)
SELECT * FROM retail_sales
VERSION AS OF 3;

In [0]:
%sql
-- avec le timestamp généré plus haut on peut récupérer les lignes de la table à un instant donné
-- il n'est pas obligatoire d'utiliser un timestamp présent dans DESCRIBE HISTORY (on peut utiliser n'importe quel timestamp sauf si supérieur à la date de la dernière opération ou inférieur à la date de création)
SELECT *
FROM retail_sales
TIMESTAMP AS OF 'COPIER LE TIMESTAMP ICI';

In [0]:
%sql
-- on restaure la table à partir de la version 1 de DESCRIBE HISTORY
-- après la mise à jour des statut et l'insertion de la nouvelle ligne
RESTORE retail_sales TO VERSION AS OF 1;

En revérifiant le nouvel historique de la table on à une nouvelle ligne avec une opération `RESTORE`.

In [0]:
%sql
DESCRIBE HISTORY retail_sales;